## Libraries

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl.worksheet._reader")

## Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

file_path = '/content/1-s2.0-S0022519315005676-mmc2.xlsx'
excel_data = pd.ExcelFile(file_path)
sheet_names = excel_data.sheet_names
data_x, data_y, time_stamps = [], [], []
for sheet in sheet_names:
    try:
        time_value = float(sheet.rstrip('h')) 
    except ValueError:
        continue

    data = pd.read_excel(file_path, sheet_name=sheet, header=None)
    x = data.iloc[2, 1:].values.astype(np.float32)
    y = data.iloc[10, 1:].values.astype(np.float32)

    data_x.extend(x)
    data_y.extend(y)
    time_stamps.extend([time_value] * len(x))

data_x = torch.tensor(np.array(data_x) / 1e3, dtype=torch.float32).view(-1, 1).to(device).requires_grad_()
data_y = torch.tensor(np.array(data_y) * 1e6, dtype=torch.float32).view(-1, 1).to(device)
time_stamps = torch.tensor(np.array(time_stamps) / 24, dtype=torch.float32).view(-1, 1).to(device).requires_grad_()

## PINN

In [ ]:
class ParameterEstimator:
    def __init__(self, K=1.7e3):
        self.D = nn.Parameter(torch.tensor(np.log(7.2), dtype=torch.float32, device=device))
        self.r = nn.Parameter(torch.tensor(np.log(1.68), dtype=torch.float32, device=device))
        self.K = torch.tensor(K/1e4, dtype=torch.float32, device=device, requires_grad=False)

    def diffusion(self, u):
        return torch.exp(self.D)

    def growth(self, u):
        return torch.exp(self.r) * u * (1 - u / torch.exp(self.K))

param_estimator = ParameterEstimator()

class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.hidden = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x, t):
        x = x.to(device).float()
        t = t.to(device).float()

        input_data = torch.cat((x, t), dim=1)
        return self.hidden(input_data)

pinn = PINN().to(device)

def compute_loss(pinn, param_estimator, data_x, data_t, data_u):
    u_pred = pinn(data_x, data_t)

    u_x = torch.autograd.grad(u_pred, data_x, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    u_t = torch.autograd.grad(u_pred, data_t, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, data_x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    d_pred = param_estimator.diffusion(u_pred)
    g_pred = param_estimator.growth(u_pred)
    lhs = u_t
    rhs = torch.autograd.grad(d_pred * u_x, data_x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0] + g_pred

    pde_residual = (lhs - rhs)

    data_loss = torch.mean((data_u/(1e6) - u_pred) ** 2)

    pde_loss = torch.mean(pde_residual ** 2)

    total_loss = data_loss +  pde_loss

    return total_loss, data_loss, pde_loss
optimizer = optim.Adam(list(pinn.parameters()) + [param_estimator.D, param_estimator.r], lr=1e-3)

epochs = 5001
loss_history = []
for epoch in range(epochs):
    optimizer.zero_grad()
    total_loss, data_loss, pde_loss = compute_loss(pinn, param_estimator, data_x, time_stamps, data_y)
    total_loss.backward()
    optimizer.step()

    loss_history.append(total_loss.item())
    if epoch % 500 == 0:
        print(f"Epoch {epoch}: Total Loss = {total_loss.item():}, Data Loss = {data_loss.item():}, PDE Loss = {pde_loss.item():}")
        
print("Final Parameter Values:")
print(f"D: {(1e4/24)*torch.exp(param_estimator.D).item():.0f}")
print(f"r: {(1/24)*torch.exp(param_estimator.r).item():.4f}")